In [2]:
import sys
import os

from collections import defaultdict

import pandas as pd
import matplotlib.pyplot as plt
import scipy.stats as ss

In [3]:
from kmer_model import KmerLinearRegressor

In [4]:
src_file = "../../predictor/regression_stability/ratios_log_stability.csv"
src_file_code = os.path.basename(src_file).partition("_")[0]
data = pd.read_csv(src_file)
data

,seq,fold,log_ratio
0,AAAAAAAAACAACAGCACCTGTCCAGGCTTCCTTAGGTACATCTTC...,train,-0.806329
1,AAAAAACTCACCCGTTTTCCTGGGATTTGTTGTAAGGAGTTTTCAC...,train,-0.798038
2,AAAAAGACATAAACTGGCACCAGTTAACTTTCTTGTACTTTTTTGC...,train,-0.843010
3,AAAAAGTAGGCGCCAATCCCCAGCACCTCTCCTCCTACTCAATTCT...,test,-0.636069
4,AAAAAGTTATGTCCATCCTGTCTTTCTACTACAGATTGCTATGAGG...,train,-0.839152
...,...,...,...
4693,TTTTGTGTTAAACTTTTACTGTTAACACACAGCATGATTTTATCCA...,train,-1.299393
4694,TTTTGTTCAGATCTCTTGAGAGTCATCCATGCAAACTCCAGATGAC...,train,-1.175334
4695,TTTTTCTGCGCTTCAGTAACAAGTGTTGGCAAACGAGACTTTCTCC...,test,-1.297885
4696,TTTTTGGTTGGGCCCAGAAAAATGTGTTAGAGTTTGTCTGTTTATC...,train,-1.189513


In [5]:
data_unique = data[["seq", "fold", "log_ratio"]].drop_duplicates()
train = data_unique[data_unique["fold"] == "train"]
test = data_unique[data_unique["fold"] == "test"]

## Defining cell types

In [6]:
cell_types = sorted(data["cell_type"].unique())
cell_types

['c1', 'c17', 'c2', 'c4', 'c6']

## Creating regression models

In [6]:
pred_results = {}
metrics_results = {}

for kmer_length in (1, 2, 3):
    metrics_results[kmer_length] = metrics = defaultdict(dict)
    regressor = KmerLinearRegressor(complement=False, kmer_length=kmer_length,
                                    linreg_kws=dict(random_state=0))
    regressor.fit(train["seq"], train["log_ratio"])
    test["pred_log_ratio"] = regressor.predict(test["seq"])
    metrics["r"] = ss.pearsonr(test["log_ratio"], test["pred_log_ratio"]).statistic
    metrics["rho"] = ss.spearmanr(test["log_ratio"], test["pred_log_ratio"]).statistic

    pred_results[("pred_log_ratio", f"k={kmer_length}")] = test["pred_log_ratio"]

/tmp/ipykernel_2369821/3470142463.py:9: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  test["pred_log_ratio"] = regressor.predict(test["seq"])
/tmp/ipykernel_2369821/3470142463.py:9: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  test["pred_log_ratio"] = regressor.predict(test["seq"])
/tmp/ipykernel_2369821/3470142463.py:9: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: h

In [10]:
pred_results_df = pd.DataFrame(pred_results).sort_index(axis=1)
pred_results_df.insert(0, "seq", test["seq"])
pred_results_df.insert(1, "log_ratio", test["log_ratio"])
pred_results_df

seq log_ratio  \
                                                                    
3     AAAAAGTAGGCGCCAATCCCCAGCACCTCTCCTCCTACTCAATTCT... -0.636069   
10    AAAAGGAAAACCGGAGGAGCAATTGGGGATCCAGGTGTCAGAGGTA... -0.893226   
27    AAACCCTTGGTGGCTGCTCAAGACAGTTCCCCTCCTAAAATCACAC... -0.874148   
38    AAAGAGCACATGGGAGCCTCCAGATGGTCCCAGTTGAGTACTGCTG... -0.621287   
43    AAAGCCAAAATATAGAACAAACACCCCACCTGCGCTATGCCAGTAA... -0.752849   
...                                                 ...       ...   
4637  TTTACCATTGGTGATAGGGTTTAATACTAAGGTTTATTTTGAGTTT... -1.166476   
4655  TTTCTCACAGCCGGTATTTATTGTCTGCTCCTCTGTGCCAGGTGCT... -0.967992   
4663  TTTCTGTTTCCAGTCCAGTTACGGACTTCCCGGCCGCCACTGGGCC... -1.018240   
4689  TTTTCTGTCACCCCAGTATCGCTGCACCCGGCCCCCCTCTCAGGCC... -1.053818   
4695  TTTTTCTGCGCTTCAGTAACAAGTGTTGGCAAACGAGACTTTCTCC... -1.297885   

     pred_log_ratio                      
                k=1       k=2       k=3  
3         -0.775879 -0.789313 -0.744746  
10        -0.800329 -0.791208 -0.805385  
27        -0.814455 -0.850833 -0.818987  
38        -0.775201 -0.774130 -0.795268  
43        -0.801939 -0.811233 -0.825711  
...             ...       ...       ...  
4637      -0.914786 -0.939015 -1.033357  
4655      -0.821887 -0.834603 -0.872448  
4663      -0.789158 -0.813912 -0.804305  
4689      -0.769731 -0.779360 -0.856130  
4695      -0.826305 -0.815120 -0.859576  

[480 rows x 5 columns]

In [11]:
pred_results_df.to_parquet("models_kmer_preds_stability.parquet")

In [12]:
import json

with open("models_kmer_stability.json", "wt") as outfile:
    json.dump(metrics_results, outfile)